trackbar for slider

use 3 track bar for RGB

show histogram changes in real time as we slide the slider


In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMG_PATH = "./images/salt_and_pepper.png"
WINDOW_NAME = "Color Balancer"
COLORS = ("b", "g", "r")

image = cv2.imread(IMG_PATH)
original_image = image.copy()


def draw_histogram(img):
    # create a blank image, same height as the original image to horizontally stack them
    hist_graph = np.zeros((image.shape[0], 520, 3), dtype=np.uint8)
    for i, color in enumerate(COLORS):
        hist = cv2.calcHist([img], [i], None, [256], [0, 256])
        # makes the count fits the hist_graph height
        height = hist_graph.shape[0]
        cv2.normalize(hist, hist, 0, height, cv2.NORM_MINMAX)
        hist = hist.flatten().astype(np.int32)
        match color:
            case "b":
                line_color = (255, 0, 0)
            case "g":
                line_color = (0, 255, 0)
            case "r":
                line_color = (0, 0, 255)
        # drawing the line
        for j in range(0, 255):
            # origin is top-left so value needs to be inverted
            # hence height-hist[j]
            cv2.line(
                hist_graph,
                (j * 2, height - hist[j]),
                ((j + 1) * 2, height - hist[j + 1]),
                line_color,
                1,
            )
    return hist_graph


def equalize_histogram(img, channel):
    # 0=No
    # 1=Gray
    # 2=B
    # 3=G
    # 4=R
    if channel == 0:
        return img
    if channel == 1:
        # greyscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        eq_gray = cv2.equalizeHist(gray)
        return cv2.cvtColor(eq_gray, cv2.COLOR_GRAY2BGR)

    equalized_img = img.copy()
    equalized_img[:, :, channel - 2] = cv2.equalizeHist(
        equalized_img[:, :, channel - 2]
    )
    return equalized_img


def apply_median_filter(img, ksize=3):
    return cv2.medianBlur(img, ksize)


def apply_mean_filter(img, ksize=3):
    return cv2.blur(img, (ksize, ksize))


def apply_gaussian_smoothing(img, ksize=3, std=0):
    # ksize must be odd and positive
    if ksize % 2 == 0:
        ksize += 1
    return cv2.GaussianBlur(img, (ksize, ksize), std)


def onTrackBar(value):
    r = cv2.getTrackbarPos("R", WINDOW_NAME) / 100
    g = cv2.getTrackbarPos("G", WINDOW_NAME) / 100
    b = cv2.getTrackbarPos("B", WINDOW_NAME) / 100
    equalizing_channel = cv2.getTrackbarPos("Equalize", WINDOW_NAME)
    filter_type = cv2.getTrackbarPos("Filter", WINDOW_NAME)

    balanced_image = original_image.astype(np.float32).copy()
    balanced_image[..., 2] *= r
    balanced_image[..., 1] *= g
    balanced_image[..., 0] *= b

    # clip the value so that anything smaller than 0 is 0 and anything larger than 255 is 255
    balanced_image = np.clip(balanced_image, 0, 255).astype(np.uint8)

    # equalizing
    balanced_image = equalize_histogram(balanced_image, equalizing_channel)

    # filtering
    if filter_type == 1:
        balanced_image = apply_median_filter(balanced_image)
    elif filter_type == 2:
        balanced_image = apply_mean_filter(balanced_image)
    elif filter_type == 3:
        balanced_image = apply_gaussian_smoothing(balanced_image)

    # draw histogram
    hist_img = draw_histogram(balanced_image)
    # stack image and histogram horizontally
    combined = np.hstack((balanced_image, hist_img))

    # Create a black bar for text (e.g., 50 px height)
    text_bar = np.zeros((150, combined.shape[1], 3), dtype=np.uint8)

    cv2.putText(
        text_bar,
        "Equalize: 0 = No, 1 = Gray, 2 = B, 3 = G, 4 = R",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        1,
        cv2.LINE_AA,
    )
    cv2.putText(
        text_bar,
        "Filter: 0 = None, 1 = Median, 2 = Mean, 3 = Gaussian",
        (10, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        1,
        cv2.LINE_AA,
    )
    cv2.putText(
        text_bar,
        "R=reset Q=quit",
        (10, 130),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        1,
        cv2.LINE_AA,
    )
    # show in one window
    final_display = np.vstack((text_bar, combined))

    cv2.imshow(WINDOW_NAME, final_display)


cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.moveWindow(WINDOW_NAME, 40, 30)

cv2.createTrackbar("R", WINDOW_NAME, 100, 200, onTrackBar)
cv2.createTrackbar("G", WINDOW_NAME, 100, 200, onTrackBar)
cv2.createTrackbar("B", WINDOW_NAME, 100, 200, onTrackBar)
cv2.createTrackbar("Equalize", WINDOW_NAME, 0, 4, onTrackBar)
cv2.createTrackbar("Filter", WINDOW_NAME, 0, 3, onTrackBar)
# dummy track bar, for some reason the last track bar is way longer than the rest
cv2.createTrackbar(" ", WINDOW_NAME, 0, 1, lambda x: None)


while True:
    key = cv2.waitKey(1) & 0xFF  # Keeping only the last 8 bits of the key code

    if key == ord("r"):  # Press 'r' to reset the balancer
        cv2.setTrackbarPos("R", WINDOW_NAME, 100)
        cv2.setTrackbarPos("G", WINDOW_NAME, 100)
        cv2.setTrackbarPos("B", WINDOW_NAME, 100)
    if key == ord("q"):  # Press 'q' to exit
        break


cv2.destroyAllWindows()